In [1]:
import sys, os, glob as _g, subprocess
try:
    import onnxruntime; print(f'ort {onnxruntime.__version__} ready')
except ImportError:
    wdirs = {os.path.dirname(w) for w in _g.glob('/kaggle/input/**/*.whl', recursive=True)
             if 'onnxruntime' in os.path.basename(w)}
    if not wdirs: raise RuntimeError('no onnxruntime wheel in /kaggle/input')
    fl = [x for d in wdirs for x in ['--find-links', d]]
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-index', *fl, 'onnxruntime'], check=True)
    import onnxruntime; print(f'ort {onnxruntime.__version__} installed')


ort 1.24.4 installed


# BirdCLEF 2026 — Pre-compute Perch Embeddings for train_audio

Runs Google Perch ONNX on every `train_audio` file and saves per-file
embeddings as `.npy` arrays, one file per recording.

**Output format**: `{primary_label}_{filename_stem}.npy`  →  shape `(N_windows, 1536)`
where each row is the Perch embedding for a consecutive 5-second window.

**Required Kaggle inputs**
1. `birdclef-2026`
2. `chiragggg/birdclef-2026-perch-onnx`  (or `rishikeshjani/perch-onnx-for-birdclef-2026`)

**Upload output as**: `chiragggg/birdclef-2026-perch-train-audio-embs`


In [ ]:
# === CELL 2: IMPORTS & CONFIG ===
import os, warnings
from pathlib import Path
import numpy as np, pandas as pd, soundfile as sf, librosa
import onnxruntime as ort
from tqdm import tqdm
warnings.filterwarnings('ignore')

CFG = dict(
    perch_sr      = 32000,
    perch_seconds = 5,
    perch_batch   = 32,       # clips per ONNX batch
    max_windows   = 6,        # cap per recording (30s max; longer clips truncated)
    max_per_species = 150,    # cap files per species to keep total manageable
)
CFG['perch_target'] = CFG['perch_sr'] * CFG['perch_seconds']  # 160000 samples

print(f"Config: perch_batch={CFG['perch_batch']}  max_windows={CFG['max_windows']}  max_per_species={CFG['max_per_species']}")


In [ ]:
# === CELL 3: PATHS & FILE SCAN ===
def _fe(*c):
    return next((p for p in c if os.path.exists(p)), c[0])

TAXONOMY_CSV  = _fe('/kaggle/input/birdclef-2026/taxonomy.csv',
                    '/kaggle/input/competitions/birdclef-2026/taxonomy.csv')
TRAIN_AUDIO   = _fe('/kaggle/input/birdclef-2026/train_audio',
                    '/kaggle/input/competitions/birdclef-2026/train_audio')
TRAIN_META    = _fe('/kaggle/input/birdclef-2026/train_metadata.csv',
                    '/kaggle/input/competitions/birdclef-2026/train_metadata.csv')
ONNX_PATH = None
for _c in [
    '/kaggle/input/birdclef-2026-perch-onnx/perch_v2_cpu.onnx',
    '/kaggle/input/datasets/chiragggg/birdclef-2026-perch-onnx/perch_v2_cpu.onnx',
    '/kaggle/input/perch-onnx-for-birdclef2026/perch_v2_cpu.onnx',
    '/kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/perch_v2.onnx',
]:
    if os.path.exists(_c):
        ONNX_PATH = _c
        break

OUT_DIR = Path('/kaggle/working/train_audio_embs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Load taxonomy for known species
taxonomy_df = pd.read_csv(TAXONOMY_CSV)
known_species = set(taxonomy_df['primary_label'].astype(str).tolist())

# Read metadata for secondary labels
meta_df = pd.read_csv(TRAIN_META)
meta_df['filename'] = meta_df['filename'].astype(str)
# Build lookup: filename_stem -> (primary_label, secondary_labels_str)
meta_lookup = {}
for _, row in meta_df.iterrows():
    stem = Path(row['filename']).stem
    meta_lookup[stem] = (str(row['primary_label']),
                         str(row.get('secondary_labels', '')))

# Scan train_audio directory, capped per species
from collections import defaultdict
species_files = defaultdict(list)
for p in sorted(Path(TRAIN_AUDIO).rglob('*.ogg')):
    sp = p.parent.name
    if sp in known_species:
        species_files[sp].append(p)

# Cap per species
file_list = []
for sp, paths in species_files.items():
    # Sort for reproducibility, take up to max_per_species
    for p in sorted(paths)[:CFG['max_per_species']]:
        file_list.append(p)

print(f'TRAIN_AUDIO : {TRAIN_AUDIO}')
print(f'ONNX_PATH   : {ONNX_PATH}')
print(f'Species found  : {len(species_files)}')
print(f'Total files    : {len(file_list)}  (capped at {CFG["max_per_species"]}/species)')
print(f'Output dir     : {OUT_DIR}')


In [ ]:
# === CELL 4: PERCH ONNX SESSION ===
_sess = None; _inp = None; _eidx = 0; _onnx_ok = False
if ONNX_PATH is None:
    print('ERROR: ONNX not found -- check Kaggle input datasets')
else:
    try:
        opts = ort.SessionOptions()
        opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        opts.intra_op_num_threads = os.cpu_count() or 4
        _sess = ort.InferenceSession(ONNX_PATH, sess_options=opts,
                                     providers=['CPUExecutionProvider'])
        _inp  = _sess.get_inputs()[0].name
        _out_names = [o.name for o in _sess.get_outputs()]
        ekey  = next((o.name for o in _sess.get_outputs()
                      if o.shape and o.shape[-1] == 1536), _out_names[0])
        _eidx = _out_names.index(ekey)
        _t    = _sess.run(None, {_inp: np.zeros((1, CFG['perch_target']), np.float32)})
        _e    = _t[_eidx]
        if _e.ndim == 3: _e = _e.mean(1)
        assert _e.shape[-1] == 1536, f'Expected 1536-d emb, got {_e.shape}'
        _onnx_ok = True
        print(f'ONNX OK: {Path(ONNX_PATH).name}  emb shape={_e.shape}')
    except Exception as ex:
        print(f'ONNX ERROR: {ex}')


In [ ]:
# === CELL 5: EMBED ALL TRAIN_AUDIO FILES ===
assert _onnx_ok, "ONNX session not ready -- check Cell 4"

def embed_file(audio_path):
    """Load audio, slice into 5s windows, run Perch ONNX, return (N_windows, 1536)."""
    try:
        y, sr = sf.read(str(audio_path), always_2d=False)
    except Exception as e:
        return None, str(e)
    if y.ndim == 2:
        y = y.mean(1)
    if sr != CFG['perch_sr']:
        y = librosa.resample(y.astype(np.float32), orig_sr=sr, target_sr=CFG['perch_sr'])
    y = y.astype(np.float32)

    n_full = len(y) // CFG['perch_target']
    n_windows = max(1, min(n_full if len(y) % CFG['perch_target'] == 0 else n_full + 1,
                           CFG['max_windows']))
    clips = []
    for i in range(n_windows):
        s = i * CFG['perch_target']
        c = y[s : s + CFG['perch_target']]
        if len(c) < CFG['perch_target']:
            c = np.pad(c, (0, CFG['perch_target'] - len(c)))
        clips.append(c)

    all_embs = []
    for bi in range(0, len(clips), CFG['perch_batch']):
        B   = np.stack(clips[bi : bi + CFG['perch_batch']])
        out = _sess.run(None, {_inp: B})[_eidx]
        if out.ndim == 3: out = out.mean(1)
        all_embs.append(out.astype(np.float32))
    return np.vstack(all_embs), None  # (N_windows, 1536)


n_done = 0; n_err = 0; n_skip = 0
err_log = []

for audio_path in tqdm(file_list, desc='embedding train_audio'):
    sp   = audio_path.parent.name
    stem = audio_path.stem
    out_name = f'{sp}_{stem}.npy'
    out_path = OUT_DIR / out_name

    if out_path.exists():
        n_skip += 1
        continue

    embs, err = embed_file(audio_path)
    if err is not None or embs is None:
        n_err += 1
        err_log.append(f'{audio_path.name}: {err}')
        continue

    np.save(str(out_path), embs)
    n_done += 1

all_npy = list(OUT_DIR.glob('*.npy'))
print(f'\nDone: {n_done} embedded  {n_skip} skipped (already exists)  {n_err} errors')
print(f'Total .npy files: {len(all_npy)}')
if err_log:
    print(f'First 5 errors: {err_log[:5]}')

# Show a sample
if all_npy:
    sample = np.load(str(all_npy[0]))
    print(f'Sample: {all_npy[0].name}  shape={sample.shape}')


In [ ]:
# === CELL 6: UPLOAD AS birdclef-2026-perch-train-audio-embs ===
import shutil, subprocess, json as _json

KAGGLE_USERNAME = os.environ.get('KAGGLE_USERNAME', 'chiragggg')
DATASET_SLUG    = 'birdclef-2026-perch-train-audio-embs'

# dataset-metadata.json sits alongside the .npy files in OUT_DIR
_meta = {
    'title':    DATASET_SLUG,
    'id':       f'{KAGGLE_USERNAME}/{DATASET_SLUG}',
    'licenses': [{'name': 'CC0-1.0'}],
}
with open(OUT_DIR / 'dataset-metadata.json', 'w') as _mf:
    _json.dump(_meta, _mf, indent=2)

all_files = list(OUT_DIR.glob('*.npy'))
print(f'Uploading {len(all_files)} .npy files from {OUT_DIR}')

_result = subprocess.run(
    ['kaggle', 'datasets', 'create', '-p', str(OUT_DIR), '--dir-mode', 'zip'],
    capture_output=True, text=True,
)
print(_result.stdout)
if _result.returncode != 0:
    print('STDERR:', _result.stderr)
    print('If dataset already exists, run:')
    print(f'  kaggle datasets version -p {OUT_DIR} -m "train_audio perch embeddings"')
else:
    print(f'Upload complete: {KAGGLE_USERNAME}/{DATASET_SLUG}')
    print('Attach in v37 training notebook as: birdclef-2026-perch-train-audio-embs')
